In [28]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import os

ROOT.gROOT.SetBatch(True)
ROOT.gErrorIgnoreLevel = ROOT.kError  # ROOT warning 억제 (필요하면 kFatal)

# ─── 0) 입력 파일 & 출력 디렉토리 ──────────────────────────────────────
# (파일 이름, 레이블)
input_samples = [
    ("hist_ST.root",   "ST_s"),
    ("hist_ST_t.root", "ST_t"),
    ("hist_ST_tW.root","ST_tW"),
    ("hist_TT.root",   "TT"),
]

out_dir = "compare_ST_like_overlay_norm"
os.makedirs(out_dir, exist_ok=True)


# ─── 1) 히스토그램 이름 리스트 얻기 (첫 파일 기준) ─────────────────────
def get_hist_names_from_root(root_file):
    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open file: {root_file}")
    plots_dir = f.Get("plots")
    if not plots_dir:
        raise RuntimeError(f"'plots' directory not found in {root_file}")

    names = []
    for key in plots_dir.GetListOfKeys():
        obj = key.ReadObj()
        # 1D 히스토그램만 (TH1 계열 & dimension=1)
        if obj.InheritsFrom("TH1") and obj.GetDimension() == 1:
            names.append(obj.GetName())
    f.Close()
    return names


hist_names = get_hist_names_from_root(input_samples[0][0])
print(f"Found {len(hist_names)} 1D histograms in 'plots/'")




Found 11 1D histograms in 'plots/'


In [29]:
# ─── 2) 각 히스토그램 이름에 대해 4개 샘플 오버레이 + 엔트리 정규화 ─────
for hname in hist_names:
    print(f"▶ {hname}")

    hist_list   = []
    labels_list = []

    edges_ref = None
    skip_this_hist = False

    # 2-1) 각 샘플에서 같은 이름의 히스토그램 가져오기
    for fname, label in input_samples:
        f = ROOT.TFile.Open(fname)
        if not f or f.IsZombie():
            print(f"   ✖ Cannot open {fname}, skip this file for {hname}")
            continue

        h = f.Get(f"plots/{hname}")
        if not h:
            f.Close()
            continue

        # 독립적으로 쓰기 위해 clone
        h_clone = h.Clone(f"{hname}_{label}_clone")
        h_clone.SetDirectory(0)
        f.Close()

        nb = h_clone.GetNbinsX()
        edges = np.array(
            [h_clone.GetBinLowEdge(i) for i in range(1, nb + 1)]
            + [h_clone.GetBinLowEdge(nb) + h_clone.GetBinWidth(nb)]
        )
        counts = np.array([h_clone.GetBinContent(i) for i in range(1, nb + 1)])

        # 첫 샘플의 binning을 기준으로, 나머지가 다 같아야 함
        if edges_ref is None:
            edges_ref = edges
        else:
            if not np.allclose(edges_ref, edges):
                print(f"   ✖ Binning mismatch in {fname} for {hname}, skip this histogram")
                skip_this_hist = True
                break

        hist_list.append(counts)
        labels_list.append(label)

    if skip_this_hist:
        continue

    # 최소 1개 이상 있어야 그림 그릴 의미
    if len(hist_list) == 0:
        print(f"   ✖ No samples for {hname}, skipping")
        continue

    # ─── 2-2) NumPy로 정리 ────────────────────────────────────────────
    hist_arr = np.array(hist_list)      # shape: (n_samples, nbins)
    edges    = edges_ref

    # ─── 2-3) 각 샘플별 총 엔트리 계산 & 가장 작은 엔트리로 정규화 ──────
    integrals = hist_arr.sum(axis=1)  # 각 샘플의 total entries
    # 양수인 것만 대상으로 최소값 계산
    positive_mask = integrals > 0
    if not np.any(positive_mask):
        print(f"   ✖ All integrals are zero for {hname}, skipping")
        continue

    min_integral = np.min(integrals[positive_mask])

    norm_hist_list = []
    norm_labels_list = []

    for counts, label, integral in zip(hist_arr, labels_list, integrals):
        if integral <= 0:
            print(f"   ⚠ {label} has zero integral for {hname}, skip this sample in plot")
            continue
        scale = float(min_integral) / float(integral)
        norm_counts = counts * scale
        norm_hist_list.append(norm_counts)
        norm_labels_list.append(label)
        # 디버그용으로 보고 싶으면:
        print(f"   {label}: integral={integral:.3g}, scale={scale:.3g}")

    # 정규화 후에도 최소 1개 이상 남아야 함
    if len(norm_hist_list) == 0:
        print(f"   ✖ No valid samples after normalization for {hname}, skipping")
        continue

    norm_hist_arr = np.array(norm_hist_list)

    # ─── 2-4) 한 개 축 위에 overlay ────────────────────────────────────
    plt.style.use(hep.style.CMS)
    fig, ax = plt.subplots(
        nrows=1,
        figsize=(10, 6),
        dpi=150,
    )
    color_map = {
        "ST_s":  "red",
        "ST_t":  "tab:orange",
        "ST_tW": "tab:green",
        "TT":    "tab:purple",
    }

    
    for counts, label in zip(norm_hist_arr, norm_labels_list):
        hep.histplot(
            counts,
            bins=edges,
            histtype="step",  # 스택 아님, 라인만
            label=label,
            ax=ax,
            color=color_map.get(label, None),
        )

    #ax.set_ylabel("Events (normalized)")
    ax.set_ylabel("Events")
    #ax.set_xlabel(hname)
    
    # 수정
    xlabel_map = {
    "b_pt":     r"$p_T(b)$ [GeV]",
    "lep_pt":   r"$p_T(\ell)$ [GeV]",
    "lep_eta":  r"$\eta(\ell)$",
    "top_pt":   r"$p_T(t)$ [GeV]",
    "top_eta":  r"$\eta(t)$",
    "MET":      r"$p_T^{\mathrm{miss}}$ [GeV]",
    "b_eta":    r"$\eta(b)$",
    "eta_b_l":  r"|$\eta(b)$-$\eta(\ell)$|"  ,
    "eta_b_t":  r"|$\eta(b)$-$\eta(t)$|"  ,
    "phi_b_t":  r"$\Delta\phi(b,t)$"  ,
    "top_mass": r"$m_t$",
    # 필요하면 계속 추가
    }
    label = xlabel_map.get(hname, hname)  # 매핑 없으면 원래 이름 유지
    ax.set_xlabel(label)
    #hep.cms.label("Private Work", data=True, lumi=59.8, year=2018, ax=ax)
    hep.cms.label("Private Work", data=True, year=2018, ax=ax)
    # CMS 스타일 라벨 (mplhep cms.label 대신 직접)
    #ax.text(
    #    0.02, 0.95,
    #    "CMS Private Work",           # 필요하면 변경
    #    transform=ax.transAxes,
    #    ha="left", va="top",
    #    fontsize=14, fontweight="bold"
    #)

    ax.legend(
        loc="upper right",
        prop={"size": 21},
        handletextpad=0.2,
        labelspacing=0.3,
        columnspacing=0.5,
    )

    # ─── 2-5) 저장 ──────────────────────────────────────────────────
    out_path = os.path.join(out_dir, f"compare_{hname}_norm.png")
    fig.savefig(out_path)
    plt.close(fig)

print("✅ All overlay *normalized* plots saved in", out_dir)

▶ MET
   ST_s: integral=1.59e+06, scale=0.16
   ST_t: integral=8.31e+05, scale=0.306
   ST_tW: integral=2.54e+05, scale=1
   TT: integral=9.21e+05, scale=0.276
▶ top_mass
   ST_s: integral=1.6e+06, scale=0.16
   ST_t: integral=8.31e+05, scale=0.307
   ST_tW: integral=2.55e+05, scale=1
   TT: integral=1.81e+06, scale=0.141
▶ top_pt
   ST_s: integral=1.59e+06, scale=0.16
   ST_t: integral=8.31e+05, scale=0.307
   ST_tW: integral=2.55e+05, scale=1
   TT: integral=1.85e+06, scale=0.138
▶ top_eta
   ST_s: integral=1.06e+06, scale=0.213
   ST_t: integral=6.47e+05, scale=0.351
   ST_tW: integral=2.27e+05, scale=1
   TT: integral=1.68e+06, scale=0.135
▶ b_pt
   ST_s: integral=1.59e+06, scale=0.16
   ST_t: integral=8.31e+05, scale=0.306
   ST_tW: integral=2.54e+05, scale=1
   TT: integral=9.23e+05, scale=0.276
▶ b_eta
   ST_s: integral=1.42e+06, scale=0.174
   ST_t: integral=7.88e+05, scale=0.312
   ST_tW: integral=2.46e+05, scale=1
   TT: integral=8.9e+05, scale=0.276
▶ lep_pt
   ST_s: integra